In [19]:
# Libraries import karo
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ML libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder,OneHotEncoder
from sklearn.feature_selection import SelectKBest, f_classif, f_regression
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import mean_squared_error,root_mean_squared_error,mean_absolute_error ,r2_score

print("All libraries imported successfully ")

All libraries imported successfully 


In [20]:
# Dataset load karo
df = pd.read_csv('../data/processed/gigsmart_dataset.csv')

print("Dataset loaded ")
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 3 rows:")
df.head(3)

Dataset loaded 
Shape: (45593, 16)

Columns: ['hour', 'day_of_week', 'is_weekend', 'is_festival', 'weather', 'zone_type', 'is_lunch_time', 'is_dinner_time', 'distance_km', 'estimated_time_min', 'app_name', 'expected_payout', 'fuel_cost', 'time_cost', 'net_profit', 'best_app']

First 3 rows:


,hour,day_of_week,is_weekend,is_festival,weather,zone_type,is_lunch_time,is_dinner_time,distance_km,estimated_time_min,app_name,expected_payout,fuel_cost,time_cost,net_profit,best_app
0,11,5,1,0,clear,normal,0,0,4.2,24,Swiggy,101.55,21.0,36.0,44.55,Swiggy
1,19,4,0,0,rain,low,0,1,14.5,33,Blinkit,220.01,72.5,49.5,98.01,Swiggy
2,8,5,1,0,fog,normal,0,0,6.4,26,Blinkit,104.92,32.0,39.0,33.92,Swiggy


## Categorical Encoding

ML models cannot understand text values.
We convert categorical columns to numbers
using One-hot Encoding.


In [21]:
# One-Hot Encoding

# Weather encoded 
weather_dummies = pd.get_dummies(df['weather'], prefix='weather')
print("Weather encoding:")
print(weather_dummies.head())

# Zone type encoded 
zone_dummies = pd.get_dummies(df['zone_type'], prefix='zone')
print("\nZone type encoding:")
print(zone_dummies.head())

# App name encoded
app_dummies = pd.get_dummies(df['app_name'], prefix='app')
print("\nApp name encoding:")
print(app_dummies.head())

# Original columns drop and new columns added
df = pd.concat([df, weather_dummies, zone_dummies, app_dummies], axis=1)
df.drop(['weather', 'zone_type', 'app_name'], axis=1, inplace=True)

# Target variable (best_app) -> Label Encoding 

le_target = LabelEncoder()
df['best_app'] = le_target.fit_transform(df['best_app'])

print("\nBest app encoding (TARGET):")
print(dict(zip(le_target.classes_,
               le_target.transform(le_target.classes_))))

print("\nEncoding done")
print(df.head(3))

print(df.shape)

Weather encoding:
   weather_clear  weather_fog  weather_rain
0           True        False         False
1          False        False          True
2          False         True         False
3           True        False         False
4           True        False         False

Zone type encoding:
   zone_busy  zone_low  zone_normal
0      False     False         True
1      False      True        False
2      False     False         True
3      False     False         True
4       True     False        False

App name encoding:
   app_Blinkit  app_Swiggy  app_Zomato
0        False        True       False
1         True       False       False
2         True       False       False
3        False        True       False
4        False       False        True

Best app encoding (TARGET):
{'Blinkit': np.int64(0), 'Swiggy': np.int64(1), 'Zomato': np.int64(2)}

Encoding done
   hour  day_of_week  is_weekend  is_festival  is_lunch_time  is_dinner_time  \
0    11            5           1

## Selecting Input and Target Features
Selecting input features (X) and target variables (y) for both machine learning tasks.

#Target variables

- **Classification Target:** `best_app`  
- **Regression Target:** `net_profit`

#Input Features

All remaining useful columns are used as input features after removing target columns.

#Purpose

- X contains independent variables used for prediction  
- y_classification used to predict best app  
- y_regression used to predict expected profit

In [22]:


# Features (after one-hot encoding already done)
X = df.drop(columns=['best_app', 'net_profit'])

# Classification target
y_class = df['best_app']

# Regression target
y_reg = df['net_profit']

print("Initial Feature Shape:", X.shape)
print("Features:", X.columns.tolist())


Initial Feature Shape: (45593, 20)
Features: ['hour', 'day_of_week', 'is_weekend', 'is_festival', 'is_lunch_time', 'is_dinner_time', 'distance_km', 'estimated_time_min', 'expected_payout', 'fuel_cost', 'time_cost', 'weather_clear', 'weather_fog', 'weather_rain', 'zone_busy', 'zone_low', 'zone_normal', 'app_Blinkit', 'app_Swiggy', 'app_Zomato']


## Train Test Split

We split the dataset into training and testing sets for both tasks:

- **Classification:** Predict best_app  
- **Regression:** Predict net_profit  

### Purpose

- Training set → used to train models  
- Testing set → used to evaluate model performance  
- Test size = 20%  
- Random state = 42 for reproducibility

In [23]:
from sklearn.model_selection import train_test_split

# Input Features
X = df.drop(columns=['best_app', 'net_profit'])

# Classification Target
y_class = df['best_app']

# Regression Target
y_reg = df['net_profit']

# Classification Split
X_train_class, X_test_class, y_train_class, y_test_class = train_test_split(
    X,
    y_class,
    test_size=0.20,
    random_state=42,
    stratify=y_class
)

# Regression Split
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X,
    y_reg,
    test_size=0.20,
    random_state=42
)

print("Classification Split:")
print("X_train:", X_train_class.shape)
print("X_test :", X_test_class.shape)
print("y_train:", y_train_class.shape)
print("y_test :", y_test_class.shape)

print("\nRegression Split:")
print("X_train:", X_train_reg.shape)
print("X_test :", X_test_reg.shape)
print("y_train:", y_train_reg.shape)
print("y_test :", y_test_reg.shape)

Classification Split:
X_train: (36474, 20)
X_test : (9119, 20)
y_train: (36474,)
y_test : (9119,)

Regression Split:
X_train: (36474, 20)
X_test : (9119, 20)
y_train: (36474,)
y_test : (9119,)


# Feature selection 

In [24]:
selector_class = SelectKBest(score_func=f_classif, k='all')

X_train_class_selected = selector_class.fit_transform(X_train_class, y_train_class)
X_test_class_selected = selector_class.transform(X_test_class)

# Feature scores
class_scores = pd.DataFrame({
    'Feature': X.columns,
    'Score': selector_class.scores_
}).sort_values(by='Score', ascending=False)

print("\nTop Classification Features:")
print(class_scores.head(10))
selector_reg = SelectKBest(score_func=f_regression, k='all')

X_train_reg_selected = selector_reg.fit_transform(X_train_reg, y_train_reg)
X_test_reg_selected = selector_reg.transform(X_test_reg)

reg_scores = pd.DataFrame({
    'Feature': X.columns,
    'Score': selector_reg.scores_
}).sort_values(by='Score', ascending=False)

print("\nTop Regression Features:")
print(reg_scores.head(10))


Top Classification Features:
            Feature         Score
16      zone_normal  24929.008322
2        is_weekend  16197.394255
14        zone_busy  10964.631270
1       day_of_week   7478.449823
13     weather_rain   4450.590924
19       app_Zomato   2059.831442
8   expected_payout   2058.701265
4     is_lunch_time   1903.993384
18       app_Swiggy   1864.066773
15         zone_low   1640.441445

Top Regression Features:
               Feature         Score
8      expected_payout  65260.175277
6          distance_km  23927.402614
9            fuel_cost  23927.402614
13        weather_rain  15191.201985
15            zone_low  15115.871242
17         app_Blinkit  10535.804359
14           zone_busy   9530.302477
10           time_cost   8938.840944
7   estimated_time_min   8938.840944
19          app_Zomato   4622.158683


 ## Classification (Target: best_app)
 
The feature selection results show that location and timing factors play the most important role in predicting the best app.

The choice of the best app is mainly influenced by where and when the delivery happens, rather than the app itself.
This suggests that external conditions (zone, day, weekend) are more critical in decision-making.

## Regression (Target: net_profit)

The results indicate that profit is primarily driven by earnings and cost-related factors.

Profit depends mostly on how much you earn vs. how much it costs to complete the delivery.
Distance, fuel, and time are critical because they directly impact expenses.


# Selecting Final Features and removing redundant features

In [25]:
top_class_features = class_scores['Feature'].head(10).tolist()
top_reg_features = reg_scores['Feature'].head(10).tolist()

X_train_class = X_train_class[top_class_features]
X_test_class = X_test_class[top_class_features]

X_train_reg = X_train_reg[top_reg_features]
X_test_reg = X_test_reg[top_reg_features]

X_train_reg = X_train_reg.drop(columns=['fuel_cost', 'time_cost'])
X_test_reg = X_test_reg.drop(columns=['fuel_cost', 'time_cost'])

print("\nTop features selected for classification:", top_class_features)
print("\nTop features selected for regression:", top_reg_features)



Top features selected for classification: ['zone_normal', 'is_weekend', 'zone_busy', 'day_of_week', 'weather_rain', 'app_Zomato', 'expected_payout', 'is_lunch_time', 'app_Swiggy', 'zone_low']

Top features selected for regression: ['expected_payout', 'distance_km', 'fuel_cost', 'weather_rain', 'zone_low', 'app_Blinkit', 'zone_busy', 'time_cost', 'estimated_time_min', 'app_Zomato']


# Classification Mode

In [26]:
clf_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000)
}

clf_results = {}

for name, model in clf_models.items():
    model.fit(X_train_class, y_train_class)
    y_pred = model.predict(X_test_class)
    
    clf_results[name] = {
        "Accuracy": accuracy_score(y_test_class, y_pred),
        "Precision": precision_score(y_test_class, y_pred, average='weighted', zero_division=0),
        "Recall": recall_score(y_test_class, y_pred, average='weighted'),
        "F1 Score": f1_score(y_test_class, y_pred, average='weighted')
    }

clf_df = pd.DataFrame(clf_results).T
print("\nClassification Results:")
print(clf_df)


Classification Results:
                     Accuracy  Precision    Recall  F1 Score
Logistic Regression  0.987937   0.987985  0.987937  0.987952


# Regression Model

In [27]:
reg_models = {
    "Linear Regression": LinearRegression()   
}

reg_results = {}

for name, model in reg_models.items():
    model.fit(X_train_reg, y_train_reg)
    y_pred = model.predict(X_test_reg)
    
    reg_results[name] = {
        "MSE": mean_squared_error(y_test_reg, y_pred),
        "RMSE": root_mean_squared_error(y_test_reg, y_pred),
        "MAE": mean_absolute_error(y_test_reg, y_pred),
        "R2": r2_score(y_test_reg, y_pred)
    }

reg_df = pd.DataFrame(reg_results).T
print("\nRegression Results:")
print(reg_df)


Regression Results:
                            MSE          RMSE           MAE   R2
Linear Regression  5.332078e-27  7.302108e-14  5.749932e-14  1.0
